# TrustLung AI — Fairness & Bias Analysis

This notebook demonstrates:
- Subgroup performance evaluation (age, gender, smoking)
- Equalized Odds Gap computation
- Bias detection in model predictions
- Fairness dashboard visualization

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.helpers import load_config, set_seed
from src.preprocessing.data_loader import generate_synthetic_dataset, ClinicalDataPreprocessor
from src.fairness.bias_analysis import FairnessAnalyzer

set_seed(42)
config = load_config('../configs/config.yaml')
print('Setup complete.')

## 1. Simulate Predictions with Demographic Data

In [ ]:
# Generate test data
data = generate_synthetic_dataset(n_samples=400, seed=42)
n    = len(data['y_test_raw'])

# Simulate slightly biased predictions (for demo)
np.random.seed(42)
y_true = data['y_test_raw']
# Add ~15% noise to simulate a realistic model
noise_mask = np.random.rand(n) < 0.15
y_pred     = y_true.copy()
y_pred[noise_mask] = np.random.randint(0, 3, noise_mask.sum())

# Simulate demographic data
preprocessor = ClinicalDataPreprocessor()
clin_df      = preprocessor.generate_demo_clinical_data(n_samples=n, seed=99)

demo_df = pd.DataFrame({
    'age_group': pd.cut(
        clin_df['AGE'], bins=[0,40,55,70,120],
        labels=['<40','40-55','55-70','70+']
    ).astype(str),
    'gender':         clin_df['GENDER'].values,
    'smoking_status': clin_df['SMOKING'].map({1:'Non-smoker',2:'Smoker'}).fillna('Unknown').values
})

print('Demographics sample:')
print(demo_df.head())

## 2. Run Fairness Analysis

In [ ]:
analyzer = FairnessAnalyzer(
    class_names=['Normal','Benign','Malignant'],
    demographic_columns=['age_group','gender','smoking_status']
)

results = analyzer.analyze(y_true, y_pred, demo_df)

# Print per-group summary
df_report = analyzer.fairness_report_dataframe(results)
print('\nFairness Report:')
print(df_report.to_string(index=False))

## 3. Fairness Dashboard

In [ ]:
analyzer.plot_fairness_dashboard(
    results,
    save_path='../outputs/plots/fairness_dashboard_notebook.png'
)
plt.show()

## 4. Equalized Odds Gap

In [ ]:
for col in ['age_group','gender','smoking_status']:
    eog = analyzer.equalized_odds_gap(y_true, y_pred, demo_df, col=col, positive_class=2)
    print(f'\nEqualized Odds Gap — {col}:')
    for group, vals in eog.items():
        print(f"  {group:12s}: TPR={vals['tpr']:.3f}  FPR={vals['fpr']:.3f}  TPR_gap={vals['tpr_gap']:.3f}")

## 5. Subgroup Confusion Matrices

In [ ]:
analyzer.plot_subgroup_confusion_matrices(
    y_true, y_pred, demo_df, col='age_group',
    save_path='../outputs/plots/fairness_cm_age_notebook.png'
)
plt.show()